# 🏠 Fork U-House Card Asset Generator

This tool allows you to generate a consistent set of isometric house assets for the [Fork U-House Card](https://github.com/fork-u/home-assistant-card).

**Features:**
- Generates day/night variants for all 4 seasons.
- Generates weather effects (rain, snow, fog, lightning).
- Generates Gaming/Immersive modes (Cyberpunk, Matrix, etc.).
- Uses Google's **Gemini 3 Pro** (or Flash) model.
- **Free to use** with a Google AI Studio API Key.

### ⚠️ Instructions
1. **Get a Free API Key**: Go to [Google AI Studio](https://aistudio.google.com/app/apikey) and create a key.
2. **Upload Reference Photos**: You will be asked to upload photos of your house (street view, side view).
3. **Run All Cells**: Click "Runtime" > "Run all" or press `Ctrl+F9`.

In [ ]:
#@title 1. Setup & Configuration
#@markdown Enter your API Key and House Description below.

import os
import time
import zipfile
from pathlib import Path
from google.colab import files
from google.colab import userdata

# Install SDK
!pip install -q -U google-generativeai
import google.generativeai as genai

# Configuration
GOOGLE_API_KEY = "" #@param {type:"string"}
HOUSE_DESCRIPTION = "Describe your house architecture here (e.g. White two-story building with dark brown roof...)" #@param {type:"string"}

# Model Selection
MODEL_TYPE = "gemini-2.5-flash-image" #@param ["gemini-3-pro-image-preview", "gemini-2.5-flash-image"]

if not GOOGLE_API_KEY:
    print("❌ Please enter your API Key!")
else:
    genai.configure(api_key=GOOGLE_API_KEY)
    print(f"✅ Configured with model: {MODEL_TYPE}")

# Directory Setup
BASE_DIR = Path("/content/house_assets")
REF_DIR = BASE_DIR / "reference"
OUT_DIR = BASE_DIR / "output"
MASTER_DIR = BASE_DIR / "master"

for d in [REF_DIR, OUT_DIR, MASTER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
#@title 2. Upload Reference Images
#@markdown Upload 1-5 photos of your house. Ideally: 1 front view, 1 side view, 1 isometric/satellite view if available.

uploaded = files.upload()

for filename in uploaded.keys():
    # Move to reference dir
    os.rename(filename, REF_DIR / filename)
    print(f"Saved reference: {filename}")

print(f"✅ Total references: {len(list(REF_DIR.glob('*')))}")

In [ ]:
#@title 3. Define Prompts & Logic

BASE_RULES = f"""
RULES (CAMERA – MANDATORY):
- VIEW: STRICT ISOMETRIC ONLY (SimCity / The Sims style).
- Isometric = equal angles, parallel lines stay parallel.
- The house and plot must be shown in this fixed isometric projection. NEVER render from a different angle.
- MARGINS: STRICT 15% Left, 15% Right, 5% Top, 15% Bottom. The house asset must be perfectly centered within these margins, with the entire glassmorphism base visible.
- ASPECT RATIO: 3:2.

RULES (VISUAL STYLE: EPIC 2026 HIGH-FIDELITY ASSET):
- Style: Ultra-high-fidelity 3D asset, "SimCity 2030" / "The Sims 5" aesthetic.
- Lighting: DRAMATIC, CINEMATIC, HIGH CONTRAST. Intense golden hour sun shafts (day) or deep navy/cyan nocturnal tones (night).
- Effects: Volumetric God rays (ONLY ON THE ASSET), soft bloom, rich ray-traced reflections.
- Materiality: Hyper-realistic textures. A thick, multi-layered Glassmorphism base plate at the very bottom, with soil/grass on top.
- Background: Solid #121212 for perfect cutout.
- Quality: 4K RAW, maximum sharpness, zero blur.

RULES (STRICT FIDELITY TO ARCHITECTURE):
- Description: {HOUSE_DESCRIPTION}
- CONSISTENCY: Keep geometry, camera angle, and scale 100% IDENTICAL to the master reference.

CRITICAL INSTRUCTION FOR GENERATION:
You will be provided with reference images. Do NOT simply apply a filter or edit the reference.
You must RE-DRAW the scene from scratch to achieve perfect 4K RAW sharpness and the requested 3D Asset Isometric style.
Use the references ONLY for geometry, angle, and architectural details.
"""

WEATHER_RULES = {
    "sunny": "Cloudless sky: Plot highly illuminated by golden sun rays.",
    "partly_cloudy": "Partly cloudy.",
    "overcast": "Overcast and gloomy.",
    "rainy": "Heavy rain: Draw a downpour in front of the house on the plot - light reflections reflecting on the plot.",
    "snowy": "Snowing: Draw intense snowfall in front of the house on the plot - light reflections reflecting on the plot.",
    "hail": "Hailstorm: Large chunks of ice on the ground instead of snow. Random puddles/patches of water under the ice.",
    "lightning": "Thunderstorms: Draw strong lightning bolts in the sky. House strongly overexposed by lightning flashes.",
    "fog": "Fog: Draw fog, a delicate cloud or slight smoke in front of house and surroundings.",
}

GAMING_MODES = {
    "synthwave": "Synthwave style: Neon purples/pinks, grid lines, retro-future vibe. Flying DeLorean parked in front.",
    "cyberpunk": "Cyberpunk style: High-tech low-life, neon signs, rain, dark gritty atmosphere. Flying DeLorean parked in front.",
    "matrix": "Matrix style: Green code rain, digital artifacts. Half the asset in green wireframe vector style. 1990 Lamborghini Countach parked in front.",
    "mario": "Mario Bros World style: Green pipe, gold coins/stars on roof, beanstalk to sky. Mario & Luigi grilling meat in front of the house.",
    "xbox_kid": "Small astronaut kid (big head) playing Xbox on a red semi-transparent sofa in front of the house. Giant projector screen glowing with game graphics. Scattered game controllers, snack bowls, and fizzy drink cans on the ground. Neon LED strips along the porch railing. Cozy gaming session vibe.",
}

# Each themed day has a prompt and a season for correct atmosphere.
# Use "*" for season to generate all 4 seasonal variants.
THEMED_DAYS = {
    "boy_bday": {
        "prompt": "Boys birthday party theme. Letterbox wrapped in blue gift paper with a big helium balloon tied to it. Colourful bunting and streamers draped across the front of the house. Pile of wrapped presents on the doorstep. A birthday cake with lit candles on a table in the front yard. Confetti scattered on the ground. Bright, joyful celebration.",
        "season": "*",
    },
    "girl_bday": {
        "prompt": "Girls birthday party theme. Letterbox wrapped in pink gift paper with a sparkly helium balloon tied to it. Pastel rainbow bunting and ribbon streamers across the front of the house. Pile of wrapped presents with big bows on the doorstep. A tiered birthday cake with lit candles on a table in the front yard. Confetti and flower petals scattered on the ground. Bright, magical celebration.",
        "season": "*",
    },
    "australia_day": {
        "prompt": "Australia Day celebration. Oversized Australian flags draped from the roof and waving from flagpoles in the yard. Green and gold bunting across the front. Blow-up kangaroo and koala on the lawn. BBQ sizzling with snags out front, esky full of cold ones. Kids with zinc on their noses running through a sprinkler. Hills hoist clothesline with Australian flag towels. Over-the-top patriotic summer vibes.",
        "season": "summer",
    },
    "new_years": {
        "prompt": "New Years Eve celebration. Spectacular fireworks bursting in the sky above the house in gold, silver, and multicolour. Glittering fairy lights and streamers draped across the front. A champagne bottle popping on the porch with glasses. Giant glowing '2026' numbers on the roof. Party hats and noisemakers scattered around. Warm summer atmosphere with a festive electric energy.",
        "season": "summer",
    },
    "fathers_day": {
        "prompt": "Father's Day celebration. Navy blue and gold colour theme. BBQ grill smoking in the front yard, cold beers on an esky, tied balloons, 'Happy Father's Day' banner across the front. Cricket bat and footy leaning against the wall. Relaxed weekend vibe.",
        "season": "spring",
    },
    "mothers_day": {
        "prompt": "Mother's Day celebration. Pastel pink and lavender colour theme. Bouquets of flowers on the doorstep, heart-shaped wreath on the front door, 'Happy Mother's Day' banner draped across the front. Breakfast tray with tea and cupcakes on the porch. Warm golden morning light.",
        "season": "autumn",
    },
}

SEASONS = ["winter", "spring", "summer", "autumn"]
TIMES = ["day", "night"]

def get_base_prompt_for_master() -> str:
    return f"""Task: Create a "SimCity 2030" style 3D asset of the house.
STRICT ISOMETRIC VIEW. 4K RAW.
{BASE_RULES}
VISUALS: Golden hour, strong sun shafts, high contrast HDR. Base ground appropriate for the asset on thick glass base.
Specific for THIS image: Summer, Day, Sunny, bright.
"""

def get_prompt_for_variant(season, time_of_day, weather="sunny", extras="") -> str:
    season_ground = {
        "winter": "Winter atmosphere, covered in snow",
        "spring": "Spring atmosphere, blooming flowers if applicable",
        "summer": "Summer atmosphere, vibrant green vegetation if applicable",
        "autumn": "Autumn atmosphere, colorful leaves",
    }
    night_rule = "Apply night colors to plot. Lights on inside house. Illuminate driveway/path." if time_of_day == "night" else ""
    weather_desc = WEATHER_RULES.get(weather, "")
    ground_desc = season_ground.get(season, "summer atmosphere")
    
    return f"""Reference: Master Image. Same isometric angle, position, scale.
{BASE_RULES}
Specific for THIS request:
- Season: {season.capitalize()}
- Time: {time_of_day.capitalize()}
- Weather: {weather_desc}
- Atmosphere/Ground: {ground_desc}
{night_rule}
{extras}
CRITICAL: REDRAW from scratch. Do NOT watermark.
"""

def generate_image(prompt, ref_images, output_path):
    print(f"Generating {output_path.name}...")
    model = genai.GenerativeModel(MODEL_TYPE)
    contents = [prompt]
    
    # Load images
    for p in ref_images:
        if p.exists():
            from PIL import Image
            contents.append(Image.open(p))
    
    try:
        response = model.generate_content(contents)
        if response.candidates and response.candidates[0].content.parts:
            for part in response.candidates[0].content.parts:
                if hasattr(part, "inline_data") and part.inline_data:
                    from PIL import Image
                    import io
                    img = Image.open(io.BytesIO(part.inline_data.data))
                    img.save(output_path, "PNG")
                    return True
        print("No image generated.")
        return False
    except Exception as e:
        print(f"Error: {e}")
        # Simple rate limit wait
        if "429" in str(e):
            print("Rate limit hit. Waiting 60s...")
            time.sleep(60)
        return False

In [ ]:
#@title 4a. (Optional) Upload Existing Master Reference
#@markdown If you already have a `_master_reference.png` you're happy with, upload it here and **skip step 4b**.
#@markdown If you want to generate a new one from scratch, skip this cell and run 4b instead.

master_path = MASTER_DIR / "_master_reference.png"

uploaded_master = files.upload()

if uploaded_master:
    filename = list(uploaded_master.keys())[0]
    os.rename(filename, master_path)
    print(f"✅ Uploaded master saved as: {master_path}")
    from IPython.display import Image, display
    display(Image(filename=str(master_path), width=400))
else:
    print("❌ No file uploaded. Run step 4b to generate a new master.")

In [ ]:
#@title 4b. Generate New Master Reference
#@markdown Skip this if you uploaded your own master in step 4a.
#@markdown This generates the base "Summer Day" image. Verify it before proceeding!

master_path = MASTER_DIR / "_master_reference.png"

if master_path.exists():
    print("⚠️ Master already exists (uploaded in 4a). Delete it first if you want to regenerate.")
    from IPython.display import Image, display
    display(Image(filename=str(master_path), width=400))
else:
    refs = list(REF_DIR.glob("*"))
    success = generate_image(
        get_base_prompt_for_master(),
        refs,
        master_path
    )

    if success:
        from IPython.display import Image, display
        print("✅ Master Generated:")
        display(Image(filename=str(master_path), width=400))
    else:
        print("❌ Master generation failed. Check your API key or try again.")

In [ ]:
#@title 5. Generate All Variants
#@markdown Loops through all seasons, times, weather, gaming modes, and themed days. This may take a while.

master_path = MASTER_DIR / "_master_reference.png"
if not master_path.exists():
    print("❌ No Master image found. Run step 4 first.")
else:
    # Standard Variants
    for season in SEASONS:
        for time_of_day in TIMES:
            # Base
            fname = f"{season}_{time_of_day}.png"
            if not (OUT_DIR / fname).exists():
                generate_image(get_prompt_for_variant(season, time_of_day), [master_path], OUT_DIR / fname)
            
            # Weather
            weather_list = ["rainy", "fog", "lightning", "overcast"]
            if season != "summer": weather_list.append("snowy")
            if season != "winter": weather_list.append("hail")
            
            for w in weather_list:
                fname_w = f"{season}_{w}_{time_of_day}.png"
                if not (OUT_DIR / fname_w).exists():
                    generate_image(get_prompt_for_variant(season, time_of_day, w), [master_path], OUT_DIR / fname_w)

    # --- Themed Days ---

    # Traditional Xmas (Winter)
    xmas_prompts = {
        "day": "Santa sliding down roof, penguins, igloo, festive decorations.",
        "night": "Santa sliding down roof, penguins, igloo, festive decorations, glowing lights.",
    }
    for t, p in xmas_prompts.items():
        fname = f"xmas_{t}.png"
        if not (OUT_DIR / fname).exists():
            generate_image(get_prompt_for_variant("winter", t, "snowy", p), [master_path], OUT_DIR / fname)

    # Australian Xmas (Summer - Southern Hemisphere)
    australian_xmas_prompts = {
        "day": "Santa sliding down roof, Australian style, Great White boomers (kangaroos), festive decorations, bright summer sun.",
        "night": "Santa climbing walls, Australian style, Great White boomers (kangaroos), Grass tree, glowing lights, festive decorations.",
    }
    for t, p in australian_xmas_prompts.items():
        fname = f"xmas_australian_{t}.png"
        if not (OUT_DIR / fname).exists():
            generate_image(get_prompt_for_variant("summer", t, "sunny", p), [master_path], OUT_DIR / fname)

    # Halloween (Spring - Southern Hemisphere)
    halloween_prompts = {
        "day": "Halloween themed, spooky pumpkins, skeletons, witches, cobwebs, orange and black theme, eerie atmosphere.",
        "night": "Halloween themed, spooky pumpkins, skeletons, witches, cobwebs, orange and black theme, orange glowing lights, eerie fog.",
    }
    for t, p in halloween_prompts.items():
        fname = f"halloween_{t}.png"
        if not (OUT_DIR / fname).exists():
            generate_image(get_prompt_for_variant("spring", t, "fog", p), [master_path], OUT_DIR / fname)

    # Gaming Modes
    for mode, desc in GAMING_MODES.items():
        fname = f"gaming_{mode}.png"
        if not (OUT_DIR / fname).exists():
            generate_image(desc + "\n" + BASE_RULES, [master_path], OUT_DIR / fname)

    # Themed Days (day/night variants, with optional all-season support via "*")
    for theme, cfg in THEMED_DAYS.items():
        theme_seasons = SEASONS if cfg["season"] == "*" else [cfg["season"]]
        for season in theme_seasons:
            for time_of_day in TIMES:
                # Include season in filename only for multi-season themes
                if cfg["season"] == "*":
                    fname = f"themed_{theme}_{season}_{time_of_day}.png"
                else:
                    fname = f"themed_{theme}_{time_of_day}.png"
                if not (OUT_DIR / fname).exists():
                    generate_image(
                        get_prompt_for_variant(season, time_of_day, "sunny", cfg["prompt"]),
                        [master_path],
                        OUT_DIR / fname,
                    )

    print("✅ Generation Complete.")

In [ ]:
#@title 5b. Generate Xmas Calendar (Dec 1-26)
#@markdown Generates a unique silly xmas scene for EVERY day of December up to Boxing Day.
#@markdown Even days = Traditional winter xmas. Odd days = Australian summer xmas.
#@markdown Dec 23 = Girl's birthday + Australian xmas combo.
#@markdown Each day gets day/night variants (52 images total).

master_path = MASTER_DIR / "_master_reference.png"
if not master_path.exists():
    print("❌ No Master image found. Run step 4 first.")
else:
    # Unique prompt for each day - elves and Santa doing silly things
    XMAS_CALENDAR = {
        1: {
            "style": "australian",
            "prompt": "December 1st kickoff! Elves unpacking a giant shipping container of Christmas decorations on the front lawn, tinsel flying everywhere. Santa in hi-vis vest directing traffic with candy canes. A confused possum tangled in fairy lights on the letterbox. Summer heat haze.",
        },
        2: {
            "style": "traditional",
            "prompt": "Elves assembling a giant advent calendar on the front wall of the house, each door the size of a window. Santa on a ladder gluing door number 2 open with a chocolate bar poking out. Gentle snowfall beginning. Cozy warm glow from inside.",
        },
        3: {
            "style": "australian",
            "prompt": "Santa test-driving a jet ski across a massive inflatable pool in the front yard, creating a huge wave. Elves in lifeguard outfits blowing whistles from a tiny wooden tower. Christmas lights strung between pool noodles. Blazing summer sun.",
        },
        4: {
            "style": "traditional",
            "prompt": "Elves running a hot chocolate stand on the front porch, steam rising from giant mugs. Santa taste-testing with a marshmallow moustache. Gingerbread house being built beside the real house. Soft snowflakes falling.",
        },
        5: {
            "style": "australian",
            "prompt": "Elves attempting to decorate a massive gum tree in the front yard with baubles and tinsel, one elf dangling from a branch. Kookaburra laughing on the roof. Santa in budgie smugglers applying sunscreen. Fairy lights everywhere.",
        },
        6: {
            "style": "traditional",
            "prompt": "St Nicholas Day! Santa leaving golden coins in shoes lined up on the front porch. Elves polishing boots to a mirror shine. A bag of presents spilling open on the doorstep. Magical golden glow through snowy scene.",
        },
        7: {
            "style": "australian",
            "prompt": "Elves set up a slip-n-slide down the front driveway with soapy water and tinsel. Santa sliding face-first with a look of pure joy. Wombat watching from the garden bed. Garden sprinkler creating rainbows. Hot summer day.",
        },
        8: {
            "style": "traditional",
            "prompt": "Elves building an elaborate gingerbread replica of the house right next to it, comparing the two. Santa as quality inspector with a clipboard, one eyebrow raised. Candy cane fence posts lining the path. Thick snow on the ground.",
        },
        9: {
            "style": "australian",
            "prompt": "Elves hosting a Christmas carol karaoke night on the front lawn with a tiny stage and microphone. Santa belting out a tune wearing a Hawaiian shirt and Santa hat. Fairy light backdrop. Cicadas providing backup vocals. Warm summer evening.",
        },
        10: {
            "style": "traditional",
            "prompt": "Elves training the reindeer for Christmas Eve in the front yard, setting up tiny hurdles and an obstacle course. Rudolph's nose lighting up the snowy scene like a spotlight. Santa with a stopwatch and coaching whistle. Frost on the windows.",
        },
        11: {
            "style": "australian",
            "prompt": "Elves running a Christmas car wash out front, scrubbing Santa's sleigh with sponges and buckets. Sleigh covered in suds. A pelican perched on the sleigh runners. Santa relaxing in a camping chair with a cold drink. Hot midday sun.",
        },
        12: {
            "style": "traditional",
            "prompt": "Twelve drummers drumming! A parade of twelve tiny elf drummers marching in formation around the front yard. Santa conducting with a candy cane baton. Musical notes made of fairy lights on the house. Magical snowy twilight.",
        },
        13: {
            "style": "australian",
            "prompt": "Elves having a Christmas piñata party in the front yard, a giant star piñata hanging from the veranda. Blindfolded elf swinging wildly with a pool noodle. Lollies scattered on the lawn. Santa catching sweets in his hat. Bright summer afternoon.",
        },
        14: {
            "style": "traditional",
            "prompt": "Elves building an army of snowmen in the front yard, each snowman wearing a different silly hat. Santa supervising from a deckchair with a giant mug of hot cocoa. Fairy lights tangled everywhere. Festive winter wonderland.",
        },
        15: {
            "style": "australian",
            "prompt": "Santa in boardshorts and sunnies surfing on a surfboard balanced on the roof ridge. Elves having a massive water balloon fight in the front yard. Inflatable palm tree with fairy lights. Summer christmas chaos.",
        },
        16: {
            "style": "traditional",
            "prompt": "All nine reindeer parked on the roof munching carrots, their antlers tangled in fairy lights. Elves on ladders wrapping the entire house in Christmas lights like a present. Snowy rooftop chaos.",
        },
        17: {
            "style": "australian",
            "prompt": "Santa in a singlet and thongs cooking shrimp and snags on a massive BBQ out front. Elves playing backyard cricket with a candy cane bat. A kangaroo wearing a Santa hat watching from the side. Esky overflowing with cold ones.",
        },
        18: {
            "style": "traditional",
            "prompt": "Elves ice skating on a frozen pond in front of the house, one elf doing a spectacular face-plant. Santa stuck halfway down the chimney with only his boots visible, elves pulling his legs. Snow-covered rooftop.",
        },
        19: {
            "style": "australian",
            "prompt": "Elves riding kangaroos in a chaotic race around the front yard, one elf clinging on for dear life. Santa floating in a blow-up pool wearing an inflatable flamingo ring. Koala sleeping in a christmas-light-wrapped tree.",
        },
        20: {
            "style": "traditional",
            "prompt": "A full toy workshop set up on the front porch with a tiny conveyor belt. Elves frantically assembling presents, wrapping paper flying everywhere. Santa checking his naughty/nice list that unrolls down the driveway. Cozy snowy scene.",
        },
        21: {
            "style": "australian",
            "prompt": "Santa doing a massive belly flop into an inflatable pool, creating a huge splash. Elves building sandcastles on the front lawn with bucket and spade sets. A cockatoo wearing tinsel around its neck perched on the letterbox. Beach umbrellas and fairy lights.",
        },
        22: {
            "style": "traditional",
            "prompt": "Epic snowball fight between two teams of elves with snow fort barricades on each side of the front yard. Santa as referee wearing a striped shirt and blowing a whistle. Snowballs mid-flight. One elf has built a snow cannon.",
        },
        23: {
            "style": "australian",
            "prompt": "Combined girl's birthday AND Australian Christmas party. Letterbox wrapped in pink gift paper with a sparkly balloon tied to it. Birthday cake AND Christmas pudding side by side on a table. Santa delivering birthday presents riding a surfboard pulled by dolphins. Elves wearing party hats AND Santa hats. Pastel bunting mixed with Christmas tinsel. Confetti and flower petals scattered everywhere. Bright magical summer celebration.",
        },
        24: {
            "style": "traditional",
            "prompt": "Christmas Eve: Santa loading the sleigh parked in front of the house, elves doing final quality checks on presents with clipboards. Reindeer lined up and ready, Rudolph's nose glowing bright red. One elf asleep on a pile of teddy bears. Magical snowy twilight.",
        },
        25: {
            "style": "australian",
            "prompt": "Christmas morning madness! Ripped wrapping paper absolutely everywhere across the yard. Kids playing with new toys on the lawn. Santa passed out asleep in a hammock strung between trees, wearing sunnies and a paper crown. Elves lounging in deckchairs. Empty pavlova plate and bon-bon hats scattered about. Warm summer morning glow.",
        },
        26: {
            "style": "australian",
            "prompt": "Boxing Day! Elves sprawled on the lawn surrounded by empty boxes and wrapping paper, food comas in full effect. Santa in a camping chair watching cricket on a portable telly, wearing a baggy green cap. Leftover Christmas ham on a table. A bin chicken raiding the leftovers. Lazy hot summer Boxing Day vibes.",
        },
    }

    for dec_day, cfg in XMAS_CALENDAR.items():
        is_traditional = cfg["style"] == "traditional"
        season = "winter" if is_traditional else "summer"
        weather = "snowy" if is_traditional else "sunny"

        for time_of_day in TIMES:
            fname = f"xmas_dec{dec_day}_{time_of_day}.png"
            if not (OUT_DIR / fname).exists():
                generate_image(
                    get_prompt_for_variant(season, time_of_day, weather, cfg["prompt"]),
                    [master_path],
                    OUT_DIR / fname,
                )

    print("✅ Xmas Calendar Generation Complete (52 images).")

In [ ]:
#@title 6. Download Results
#@markdown Zips all generated images and triggers download.

zip_path = BASE_DIR / "house_assets.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in OUT_DIR.glob("*.png"):
        zipf.write(file, file.name)
    # Add Master too
    if master_path.exists():
        zipf.write(master_path, "_master_reference.png")

print(f"Zipped {len(list(OUT_DIR.glob('*.png')))} files.")
files.download(str(zip_path))